# Other methods for solving nonlinear equations in 1d

In [90]:
# | include: false
# useful definitions that we've used so far:

using Plots
using LaTeXStrings
using Polynomials
using PrettyTables

function simple_iteration( g, x1; N=100, tol=1e-10 )
    x = [ x1 ]
    for n in 2:N
        push!( x, g(x[n-1]) )
        if (abs(g(x[end]) - x[end]) < tol)
            break
        elseif (x[end] == Inf)
            @warn "simple iteration diverges to Inf";
            break
        elseif (x[end] == -Inf)
            @warn "simple iteration diverges to -Inf";
            break
        end 
    end
    return x
end

function relaxation( f, λ, x1; N=100, tol=1e-10)
    x = [x1]
    r = 0.;
    for n in 2:N
        push!( x, x[n-1] - λ*f(x[n-1]) )
        r = abs(f(x[end]));
        if (r < tol)
            return x
        end
    end
    @warn "max interations with |f| = $r";
    return x
end

function Newton( f, f_prime, x1; N=100, tol=1e-10)
    x = [x1]
    for n in 2:N
        push!( x, x[n-1] - f(x[n-1])/f_prime(x[n-1]) )
        r = abs(f(x[end]));
        if (r < tol)
            return x
        end
    end
    @warn "max interations |f| = $r";
    return x
end

function orderOfConvergence( x, ξ; α=0 )
    err = @. abs(x - ξ)
    logerr = @. log10( err )
    ratios = [NaN; [logerr[i+1] / logerr[i] for i in 1:length(logerr)-1]]
    if (α == 0) 
        α = ratios[end]
        αr = round(α, sigdigits=3)
    end
    mu = [NaN; [err[i+1] / err[i]^α for i in 1:length(err)-1]]
    pretty_table( 
         [1:length(x) x err ratios mu];
        column_labels = ["iteration", "x[n]", "absolute error", "alpha", "mu (α = $α)" ]
    )
end

function μ( x, ξ; α=1 )
    return @. abs( x[2:end] - ξ ) / ( abs(x[1:end-1] - ξ )^α );
end 

μ (generic function with 1 method)

Here, I have summarised some of the answers for the following question on Assignment 5: 

> Find a research paper explaining a method named after one of the following people: Halley, Householder, Osada, Ostrowski, Steffensen, Traub. What is the main novelty of this method? How does it (claim to) improve upon previous methods in the literature? Implement your chosen method and test it on a function of your choice. Please clearly cite which paper you are describing.

## Steffensen

Papers you cited: @Steffensen1933, @Jain2007, @Jaiswal2013, @Cordero2012

In Newton's method, we replace the derivative $f'(x_n)$ with the divided difference $f[x_n, x_n + f(x_n)] = \frac{ f(x_n + f(x_n)) - f(x_n) }{(x_n + f(x_n)) - x_n}$ which gives 

\begin{align*}
    x_{n+1} &= x_n  - \frac{ f(x_n) }{ f[x_n, x_n + f(x_n)] } \\
    %
    &=  x_n  - \frac{ f(x_n)^2 }{ f(x_n + f(x_n)) - f(x_n) }
\end{align*}

* Under some conditions ```[missing details]```, this method converges quadratically but now we do not need to compute the derivative of $f$ which may be costly in practice.
* The efficiency index is $\sqrt{2}$ which is the same as Newton, 

<pre>
Steffensen's method applied to f(x) = x^3 - 2x^2 - 5 starting at x[0] = 2.5
┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 2) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     2.5 │       0.190647 │     NaN │        NaN │
│       2.0 │    3.46 │       0.769353 │ 0.15821 │    21.1672 │
│       3.0 │ 3.41581 │       0.725159 │ 1.22562 │    1.22513 │
│       4.0 │ 3.36955 │       0.678903 │  1.2051 │    1.29105 │
│       5.0 │ 3.32103 │       0.630382 │ 1.19147 │    1.36769 │
│       6.0 │ 3.27002 │       0.579368 │ 1.18288 │    1.45796 │
│       7.0 │ 3.21627 │       0.525618 │ 1.17838 │    1.56589 │
│       8.0 │ 3.15953 │       0.468885 │ 1.17758 │    1.69718 │
│       9.0 │  3.0996 │       0.408956 │ 1.18055 │    1.86013 │
│      10.0 │ 3.03637 │       0.345725 │ 1.18785 │    2.06717 │
│      11.0 │ 2.97001 │       0.279367 │ 1.20065 │     2.3373 │
│      12.0 │ 2.90137 │       0.210718 │ 1.22114 │    2.69992 │
│      13.0 │ 2.83271 │       0.142066 │ 1.25316 │    3.19953 │
│      14.0 │ 2.76925 │      0.0785979 │ 1.30334 │     3.8943 │
│      15.0 │  2.7204 │      0.0297563 │ 1.38189 │    4.81679 │
│      16.0 │  2.6958 │     0.00515575 │ 1.49874 │    5.82281 │
│      17.0 │ 2.69082 │    0.000172092 │ 1.64542 │    6.47406 │
│      18.0 │ 2.69065 │     1.96084e-7 │ 1.78192 │    6.62097 │
│      19.0 │ 2.69065 │    2.54907e-13 │ 1.87753 │    6.62974 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘
</pre>

This method applied to Kepler's equation (A5 Section B) with $\epsilon = 0.9$ converges cubically (same as Newton in this particular case):

<pre>
Steffensen's method applied to f(ψ) = ψ - 0.9 sin(ψ) - 2π; starting at x[0] = 5.0
┌───────────┬─────────┬────────────────┬───────────┬────────────┐
│ iteration │    x[n] │ absolute error │     alpha │ mu (α = 3) │
├───────────┼─────────┼────────────────┼───────────┼────────────┤
│       1.0 │     5.0 │        1.28319 │       NaN │        NaN │
│       2.0 │ 5.45139 │       0.831796 │ -0.738606 │   0.393685 │
│       3.0 │    5.82 │       0.463185 │   4.17895 │   0.804829 │
│       4.0 │ 6.11414 │       0.169042 │    2.3097 │     1.7011 │
│       5.0 │ 6.26849 │      0.0146953 │    2.3741 │    3.04227 │
│       6.0 │ 6.28317 │     1.09845e-5 │   2.70578 │    3.46136 │
│       7.0 │ 6.28319 │    1.77636e-15 │   2.97435 │    1.34025 │
└───────────┴─────────┴────────────────┴───────────┴────────────┘
</pre>

### Variants of Steffensen:

Papers you cited:  @Hafiz2013, @Eskandari2022, 

Another way of doing Steffensen is the backwards difference formula:

\begin{align}
    x_{n+1} &= x_n - \frac{ f(x_n)^2 }{ f[x_n, x_n - f(x_n)] } \\
    %
    &= x_n - \frac{ f(x_n)^2 }{ f(x_n) - f(x_n - f(x_n)) } 
\end{align}

Or the central formula:

\begin{align}
    x_{n+1} &= x_n - \frac{f(x_n)}{ f[x_n - f(x_n), x_n + f(x_n)] } \\
    %
    &=  x_n - \frac{2 f(x_n)^2}{ f(x_n + f(x_n)) - f(x_n - f(x_n)) }
\end{align}

Or one could use a *convex combination*: for $\theta \in [0,1]$, 

\begin{align}
    x_{n+1} = &\theta \left( x_n  - \frac{ f(x_n)^2 }{ f(x_n + f(x_n)) - f(x_n) } \right) \\
    &+ (1-\theta) \left( x_n  - \frac{ f(x_n)^2 }{ f(x_n) - f(x_n -f(x_n) ) } \right)
\end{align}

Or one could change the step size in the standard Steffensen's method: for $\tau \in \mathbb R$, consider

\begin{align}
    x_{n+1} &= x_n  - \frac{ f(x_n) }{ f[ x_n, x_n  + \tau f(x_n) ] } \\
    %
    &= x_n  - \frac{ \tau f(x_n) }{ f(x_n  + \tau f(x_n)) - f(x_n) }
\end{align}

## Halley

Papers you cited: @Davies1975, @Brown1977, @Alefeld1981, @Gander1985, @Scavo1995, @Proinov2014, @Gnang2018, @Barrada2020

Suppose $f$ is twice continuously differentiable and consider:

\begin{align}
    x_{n+1} = x_n - \frac{ f(x_n)f'(x_n) }{ f'(x_n)^2-\frac{1}{2} f(x_n) f''(x_n)  }
\end{align}

Under some conditions ```[missing details]```, this method converges cubically. 

<pre>
Halley's method applied to f(x) = log x starting at x[0] = 5.0
┌───────────┬──────────┬────────────────┬───────────┬──────────────┐
│ iteration │     x[n] │ absolute error │     alpha │ mu (α = 3.0) │
├───────────┼──────────┼────────────────┼───────────┼──────────────┤
│       1.0 │      5.0 │            4.0 │       NaN │          NaN │
│       2.0 │ 0.541029 │       0.458971 │ -0.561762 │   0.00717142 │
│       3.0 │   1.0207 │      0.0207005 │   4.97914 │     0.214104 │
│       4.0 │ 0.999999 │      7.1683e-7 │   3.64876 │     0.080812 │
│       5.0 │      1.0 │            0.0 │       Inf │          0.0 │
└───────────┴──────────┴────────────────┴───────────┴──────────────┘
</pre>


For the following function, Newton converged linearly with asymptotic error constant $\mu = \frac{2}3$:

<pre>
Halley's method applied to f(ψ) = ψ - sin(ψ) - 2π; starting at x[0] = 6.0
┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 1) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     6.0 │       0.283185 │     NaN │        NaN │
│       2.0 │ 6.14169 │       0.141498 │ 1.54992 │   0.499667 │
│       3.0 │ 6.21245 │      0.0707375 │ 1.35455 │   0.499917 │
│       4.0 │ 6.24782 │      0.0353673 │  1.2617 │   0.499979 │
│       5.0 │  6.2655 │      0.0176834 │ 1.20741 │   0.499995 │
│       6.0 │ 6.27434 │      0.0088417 │ 1.17178 │   0.499999 │
│       7.0 │ 6.27876 │     0.00442085 │  1.1466 │        0.5 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘
</pre>

## Osada

Paper you used: @Osada1994

Suppose $f:\mathbb R \to \mathbb R$ has a root of multiplicity $m \geq 2$ and define

\begin{align}
    x_{n+1} = x_n - \frac12 m (m+1) \frac{ f(x_n) }{ f'(x_n) } + \frac12 (m-1)^2 \frac{f'(x_n)}{ f''(x_n) } 
\end{align}

* For $f(x) = (x - \xi)^m$ this iteration converges in one step since 

\begin{align}
    &x - \frac12 m (m+1) \frac{ (x-\xi)^m }{ m (x-\xi)^{m-1} } + \frac12 (m-1)^2 \frac{ m (x - \xi)^{m-1} }{ m(m-1) (x - \xi)^{m-2} } \\
    %
    &= x - \frac12  (m+1)  (x-\xi)  + \frac12 (m-1)  (x - \xi)  = \xi\\
    %
\end{align}

* Under some conditions ```[details missing]```, Osada's method converges cubically, 

<pre>
Osada's method applied to f(x) = exp( 2(x-1) ) (x-1)^3 (here, m = 3) starting at x[0] = 0.0
┌───────────┬─────────┬────────────────┬───────────┬────────────┐
│ iteration │    x[n] │ absolute error │     alpha │ mu (α = 3) │
├───────────┼─────────┼────────────────┼───────────┼────────────┤
│       1.0 │     0.0 │            1.0 │       NaN │        NaN │
│       2.0 │     7.0 │            6.0 │       Inf │        6.0 │
│       3.0 │ 5.41081 │        4.41081 │  0.828269 │  0.0204204 │
│       4.0 │ 3.93473 │        2.93473 │  0.725453 │  0.0341989 │
│       5.0 │ 2.63744 │        1.63744 │  0.458043 │  0.0647833 │
│       6.0 │ 1.63668 │        0.63668 │ -0.915547 │   0.145018 │
│       7.0 │  1.0993 │       0.099301 │   5.11552 │   0.384761 │
│       8.0 │ 1.00088 │     0.00088033 │   3.04607 │   0.899052 │
│       9.0 │     1.0 │    7.56534e-10 │   2.98531 │     1.1089 │
└───────────┴─────────┴────────────────┴───────────┴────────────┘
</pre>

<pre>
Osada's method applied to f(ψ) = ψ - sin(ψ) - 2π (here, m = 3) starting at x[0] = 6
┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 3) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     6.0 │       0.283185 │     NaN │        NaN │
│       2.0 │  6.2828 │    0.000389449 │ 6.22261 │   0.017149 │
│       3.0 │ 6.28319 │     4.24732e-9 │ 2.45542 │    71.9059 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘
</pre>

## Traub

Paper you used: @SaeedK2023

We shall consider the following method: 

\begin{align}
    y_n &= x_n - \frac{ f(x_n) }{ f'(x_n) } \\
    %
    x_{n+1} &= x_n - \frac{ f(x_n) }{ \frac{1}{2} [ f'(x_n) + f'(y_n) ] }
\end{align}

Under some conditions ```[details missing]```, this method has cubic order of convergence.

Applied to the function $f(x) = x^3 - 8$, Newton's method converges quadratically

<pre>
Newton's method applied to f(x) = x^3 - 8 starting at x[0] = 100.0
┌───────────┬─────────┬────────────────┬───────────┬────────────┐
│ iteration │    x[n] │ absolute error │     alpha │ mu (α = 2) │
├───────────┼─────────┼────────────────┼───────────┼────────────┤
│       1.0 │   100.0 │           98.0 │       NaN │        NaN │
│       2.0 │ 66.6669 │        64.6669 │   0.90933 │ 0.00673333 │
│       3.0 │ 44.4452 │        42.4452 │  0.899014 │    0.01015 │
│       4.0 │ 29.6315 │        27.6315 │  0.885477 │  0.0153372 │
│       5.0 │ 19.7574 │        17.7574 │  0.866779 │  0.0232579 │
│       6.0 │ 13.1784 │        11.1784 │  0.839121 │  0.0354505 │
│       7.0 │ 8.80096 │        6.80096 │  0.794149 │  0.0544265 │
│       8.0 │ 5.90174 │        3.90174 │   0.71016 │  0.0843562 │
│       9.0 │ 4.01105 │        2.01105 │  0.513183 │   0.132101 │
│      10.0 │ 2.83978 │       0.839784 │ -0.249923 │   0.207645 │
│      11.0 │ 2.22386 │       0.223862 │    8.5718 │   0.317428 │
│      12.0 │ 2.02178 │      0.0217786 │    2.5568 │    0.43458 │
│      13.0 │ 2.00023 │    0.000233757 │    2.1849 │   0.492838 │
│      14.0 │     2.0 │     2.73168e-8 │   2.08292 │   0.499922 │
│      15.0 │     2.0 │    4.44089e-16 │    2.0298 │   0.595128 │
└───────────┴─────────┴────────────────┴───────────┴────────────┘
</pre>

whereas Traub's method converges cubically

<pre>
Traub's method applied to f(x) = x^3 - 8 starting at x[0] = 100.0
┌───────────┬─────────┬────────────────┬───────────┬─────────────┐
│ iteration │    x[n] │ absolute error │     alpha │  mu (α = 3) │
├───────────┼─────────┼────────────────┼───────────┼─────────────┤
│       1.0 │   100.0 │           98.0 │       NaN │         NaN │
│       2.0 │ 53.8466 │        51.8466 │  0.861138 │  5.50861e-5 │
│       3.0 │  28.996 │         26.996 │  0.834713 │ 0.000193704 │
│       4.0 │  15.619 │         13.619 │  0.792388 │ 0.000692223 │
│       5.0 │    8.43 │           6.43 │  0.712617 │  0.00254553 │
│       6.0 │ 4.60695 │        2.60695 │  0.514881 │  0.00980617 │
│       7.0 │ 2.70353 │       0.703532 │ -0.366989 │   0.0397087 │
│       8.0 │  2.0505 │      0.0504967 │   8.49115 │    0.145015 │
│       9.0 │ 2.00004 │     3.55831e-5 │   3.43073 │    0.276347 │
│      10.0 │     2.0 │    1.33227e-14 │   3.11894 │    0.295707 │
└───────────┴─────────┴────────────────┴───────────┴─────────────┘
</pre>

## Ostrowsky

See [Ostrowsky](Ostrowsky-KhoiDuong.html) for more details.

Paper you used: @ChristianBeleaPostigo2023

\begin{align}
    y_n &= x_n - \frac{f(x_n)}{f'(x_n)} \\
    x_{n+1} &= y_n - \frac{ f(y_n) }{ f'(y_n) } \frac{ f(x_n) }{ f(x_n) - 2f(y_n) }
\end{align}

* Suppose $f(\xi) = 0$ and $f$ is three times continuously differentiable with $f'(\xi) \not= 0$. Then, if $|x_1 - \xi|$ is sufficiently small, Ostrowsky's method converges quartically (with order $4$).
* The efficiency index of Newton is $2^{\frac12} \approx 1.41...$, whereas the efficiency index of Ostrowsky is $4^{\frac13} \approx 1.59...$

## Householder

Paper used: @Foo2025

Fix $d \in \mathbb N$. Suppose $f$ is $d+1$ times continuously differentiable and define

\begin{align*}
    x_{n+1} = x_n + d 
    \frac
        { (1/f)^{(d-1)}(x_n) }
        { (1/f)^{(d)}(x_n) }.
\end{align*}

Under some conditions [missing details], this method converges with order of convergence $d+1$. 


<div class='alert alert-block alert-danger'><b>Remark.</b> 

When $d = 1$, we have 

\begin{align*}
    d 
    \frac
        { (1/f)^{(d-1)}(x_n) }
        { (1/f)^{(d)}(x_n) }
    %
    &= 1 \frac{ (1/f)(x_n) }{ -(f'/f^2)(x_n) } \\
    %
    &= - \frac{f(x_n)}{f'(x_n)} 
\end{align*}

and so Householder with $d = 1$ is the same as Newton's method.

</div> 

<div class='alert alert-block alert-danger'><b>Remark.</b> 

When $d = 2$, we have 

\begin{align*}
    d 
    \frac
        { (1/f)^{(d-1)}(x_n) }
        { (1/f)^{(d)}(x_n) }
    %
    &= 2 \frac{ (-f'/f^2)(x_n) }{ ((-f''f +2 (f')^2 )/f^3)(x_n) } \\
    %
    &=  -2 \frac{ (ff')(x_n) }{ (-f''f +2 (f')^2 )(x_n) } \\
    %
    &=  - \frac{ f(x_n)f'(x_n) }{ f'(x_n)^2-\frac{1}{2} f(x_n) f''(x_n)  }
\end{align*}

and so Householder with $d = 2$ is the same as Halley's method.

</div> 

## References

In [ ]:
# | include: false
# Kresten

f = x -> log(x);
f_prime = x -> 1/x
f_2prime = x-> -1/x^2

tol = 1e-10;
N = 100;
x = [5.0];

for i in 1:N
    x_n = x[end];
    if abs(f(x_n)) < tol
        break
    else
        push!(x, x_n - f(x_n)*f_prime(x_n)/(f_prime(x_n)^2 - 0.5*f(x_n)*f_2prime(x_n)));
    end
end

println( "Halley's method applied to f(x) = log x starting at x[0] = 5.0" )
orderOfConvergence(x, 1.0; α=3.0)

Halley's method applied to f(x) = log x starting at x[0] = 5.0
┌───────────┬──────────┬────────────────┬───────────┬──────────────┐
│ iteration │     x[n] │ absolute error │     alpha │ mu (α = 3.0) │
├───────────┼──────────┼────────────────┼───────────┼──────────────┤
│       1.0 │      5.0 │            4.0 │       NaN │          NaN │
│       2.0 │ 0.541029 │       0.458971 │ -0.561762 │   0.00717142 │
│       3.0 │   1.0207 │      0.0207005 │   4.97914 │     0.214104 │
│       4.0 │ 0.999999 │      7.1683e-7 │   3.64876 │     0.080812 │
│       5.0 │      1.0 │            0.0 │       Inf │          0.0 │
└───────────┴──────────┴────────────────┴───────────┴──────────────┘


In [ ]:
# | include: false 
# Roy

function Halley( f, f_prime, f_doubleprime, x1; N=100, tol=1e-10)
    x = [x1]
    for n in 2:N
        push!( x, x[n-1] - (f(x[n-1])/f_prime(x[n-1]))*(1/(1-((f_doubleprime(x[n-1])*f(x[n-1]))/(f_prime(x[n-1]))^2)/2)))
        r = abs(f(x[end]));
        if (r < tol)
            return x
        end
    end
    @warn "max interations |f| = $r";
    return x
end

f = x -> x - sin(x) - 2π
f_prime = x -> 1 - cos(x)
f_doubleprime = x -> sin(x)

h = Halley(f, f_prime, f_doubleprime, 6.0, tol=1e-7)
orderOfConvergence(h, 2π; α = 1)

┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 1) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     6.0 │       0.283185 │     NaN │        NaN │
│       2.0 │ 6.14169 │       0.141498 │ 1.54992 │   0.499667 │
│       3.0 │ 6.21245 │      0.0707375 │ 1.35455 │   0.499917 │
│       4.0 │ 6.24782 │      0.0353673 │  1.2617 │   0.499979 │
│       5.0 │  6.2655 │      0.0176834 │ 1.20741 │   0.499995 │
│       6.0 │ 6.27434 │      0.0088417 │ 1.17178 │   0.499999 │
│       7.0 │ 6.27876 │     0.00442085 │  1.1466 │        0.5 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘


In [ ]:
# | include: false 

# Evan 
g = x -> x^3 - 2x^2 - 5;
function Steffensen(f, x1; N=100, tol=1e-10)
    x = [x1]
    for n in 2:N
        push!(x, x[n-1] - f(x[n-1])^2 / (f(x[n-1] + f(x[n-1])) - f(x[n-1])))
        r = abs(f(x[end]));
        if (r < tol)
            return x
        end
    end
    @warn "max iterations |f| = $r";
    return x
end

it_g = Steffensen(g, 2.5, N=100, tol=1e-10);
println( "Steffensen's method applied to f(x) = x^3 - 2x^2 - 5 starting at x[0] = 2.5" )
orderOfConvergence( it_g, 2.69064744802861375035078888267680615180196955391244111955922589044486470682, α=2 )

Steffensen's method applied to f(x) = x^3 - 2x^2 - 5 starting at x[0] = 2.5
┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 2) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     2.5 │       0.190647 │     NaN │        NaN │
│       2.0 │    3.46 │       0.769353 │ 0.15821 │    21.1672 │
│       3.0 │ 3.41581 │       0.725159 │ 1.22562 │    1.22513 │
│       4.0 │ 3.36955 │       0.678903 │  1.2051 │    1.29105 │
│       5.0 │ 3.32103 │       0.630382 │ 1.19147 │    1.36769 │
│       6.0 │ 3.27002 │       0.579368 │ 1.18288 │    1.45796 │
│       7.0 │ 3.21627 │       0.525618 │ 1.17838 │    1.56589 │
│       8.0 │ 3.15953 │       0.468885 │ 1.17758 │    1.69718 │
│       9.0 │  3.0996 │       0.408956 │ 1.18055 │    1.86013 │
│      10.0 │ 3.03637 │       0.345725 │ 1.18785 │    2.06717 │
│      11.0 │ 2.97001 │       0.279367 │ 1.20065 │     2.3373 │
│      12.0 │ 2.90137 │     

In [ ]:
# | include: false 
# nicole

ϵ = 0.9;
f = ψ -> ψ - ϵ * sin(ψ) - 2π;
x1 = 5.0;

xs= Steffensen(f, x1)
println( "Steffensen's method applied to f(ψ) = ψ - 0.9 sin(ψ) - 2π; starting at x[0] = 5.0" )
orderOfConvergence( xs, 2π, α=3 )

Steffensen's method applied to f(ψ) = ψ - 0.9 sin(ψ) - 2π; starting at x[0] = 5.0
┌───────────┬─────────┬────────────────┬───────────┬────────────┐
│ iteration │    x[n] │ absolute error │     alpha │ mu (α = 3) │
├───────────┼─────────┼────────────────┼───────────┼────────────┤
│       1.0 │     5.0 │        1.28319 │       NaN │        NaN │
│       2.0 │ 5.45139 │       0.831796 │ -0.738606 │   0.393685 │
│       3.0 │    5.82 │       0.463185 │   4.17895 │   0.804829 │
│       4.0 │ 6.11414 │       0.169042 │    2.3097 │     1.7011 │
│       5.0 │ 6.26849 │      0.0146953 │    2.3741 │    3.04227 │
│       6.0 │ 6.28317 │     1.09845e-5 │   2.70578 │    3.46136 │
│       7.0 │ 6.28319 │    1.77636e-15 │   2.97435 │    1.34025 │
└───────────┴─────────┴────────────────┴───────────┴────────────┘


In [157]:
# | include: false
f = x -> cos(x) - x
xs= Steffensen(f, 1.0)
println( "Steffensen's method applied to f(x) = cos(x) - x starting at x[0] = 1.0" )
orderOfConvergence( xs, 0.7390851332151606416553120876738734040134117589007574649656806357732846548835475945993761069317665318498012466439871630277149036913084203157804405746207786885249 , α=2 )

Steffensen's method applied to f(x) = cos(x) - x starting at x[0] = 1.0
┌───────────┬──────────┬────────────────┬─────────┬────────────┐
│ iteration │     x[n] │ absolute error │   alpha │ mu (α = 2) │
├───────────┼──────────┼────────────────┼─────────┼────────────┤
│       1.0 │      1.0 │       0.260915 │     NaN │        NaN │
│       2.0 │  0.72801 │      0.0110748 │  3.3516 │   0.162681 │
│       3.0 │ 0.739067 │     1.81663e-5 │  2.4241 │   0.148114 │
│       4.0 │ 0.739085 │    4.90851e-11 │ 2.17457 │   0.148736 │
└───────────┴──────────┴────────────────┴─────────┴────────────┘


In [118]:
# | include: false

# Braydon

function Steffenson_Hybrid( f, x1, w; N=100, tol=1e-10)
    if (w > 1) || (w < 0)
        return 0
    end
    x = [x1]
    for n in 2:N
        prev = x[n-1]
        # paper includes a 2.0 scalar at beginning which I removed because it was leading to divergence.
        sol = prev - (f(prev))^2.0 * ( (w / ( f(prev + f(prev)) - f(prev) ) ) + ( (1.0-w) / (f(prev) - f(prev-f(prev)) ) ) )

        push!( x, sol )
        if (abs(f(x[end])) < tol)
            return x
        end
    end
    return x
end

ξ_test = 1.261556999
test_f = x -> x^3 -exp(x) + 2x - 1
# w 
x = Steffenson_Hybrid(test_f,3.0,0.25)
orderOfConvergence( x, 1.2615569990915; α = 2 )

┌───────────┬─────────┬────────────────┬───────────┬────────────┐
│ iteration │    x[n] │ absolute error │     alpha │ mu (α = 2) │
├───────────┼─────────┼────────────────┼───────────┼────────────┤
│       1.0 │     3.0 │        1.73844 │       NaN │        NaN │
│       2.0 │ 2.85597 │        1.59442 │  0.843611 │   0.527571 │
│       3.0 │ 2.68505 │        1.42349 │  0.756925 │   0.559952 │
│       4.0 │ 2.46725 │         1.2057 │  0.529743 │   0.595017 │
│       5.0 │ 2.15044 │       0.888888 │ -0.629664 │   0.611463 │
│       6.0 │ 1.54591 │       0.284358 │   10.6765 │   0.359891 │
│       7.0 │ 1.17008 │      0.0914791 │   1.90187 │    1.13134 │
│       8.0 │ 1.26026 │     0.00130111 │   2.77823 │   0.155478 │
│       9.0 │ 1.26156 │     6.49533e-7 │   2.14417 │   0.383684 │
│      10.0 │ 1.26156 │     6.4837e-14 │   2.13146 │   0.153681 │
└───────────┴─────────┴────────────────┴───────────┴────────────┘


In [148]:
# | include: false
# Antonio

# Implementation

function osada(f, fp, fpp, m, x1; N=100, tol=1e-14)
    x = [x1]
    for k in 2:N
        xn = x[end]
        fpn = fp(xn); fppn = fpp(xn)
        if fppn == 0
            @warn "f''(x) is zero at x = $xn; iteration cannot proceed"
            break
        end
        un = f(xn) / fpn
        step = 0.5*m*(m+1)*un - 0.5*(m-1)^2 * (fpn / fppn)
        xnext = xn - step
        push!(x, xnext)
        if abs(f(xnext)) < tol
            break
        end
    end
    return x
end

m  = 7
ξ  = 1.0
a  = 2            
x1 = 2.0 # ξ + 8.0        

g = x -> exp(2*(x-ξ)) * (x - ξ)^m 
gp = x -> exp(2*(x-ξ)) * ( m*(x-ξ)^(m-1) + 2*(x-ξ)^m )
gpp = x -> exp(2*(x - ξ)) * ( m*(m-1)*(x-ξ)^(m-2) + 4*m*(x-ξ)^(m-1) + 4*(x-ξ)^m )

xs = osada(g, gp, gpp, m, x1; N=50, tol=1e-14)

println("Osada's method applied to f(x) = exp( 2(x-1) ) (x-1)^3 starting at x[0] = 0.0")
orderOfConvergence(xs, ξ; α=3)   

Osada's method applied to f(x) = exp( 2(x-1) ) (x-1)^3 starting at x[0] = 0.0
┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 3) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     2.0 │            1.0 │     NaN │        NaN │
│       2.0 │ 1.07808 │      0.0780781 │    -Inf │  0.0780781 │
│       3.0 │ 1.00007 │     6.71361e-5 │ 3.76808 │   0.141049 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘


In [ ]:
# | include: false 
f = ψ -> ψ - sin(ψ) - 2π
df = ψ -> 1 - cos(ψ)
d2f = ψ -> sin(ψ)
m = 3 # multiplicity of 3 since f'''(ξ) = 1 != 0

x1_osada = osada(f, df, d2f, m, 6., tol=1e-100)
println("Osada's method applied to f(ψ) = ψ - sin(ψ) - 2π starting at x[0] = 6")
orderOfConvergence(x1_osada, 2π, α=3)

Osada's method applied to f(ψ) = ψ - sin(ψ) - 2π starting at x[0] = 6
┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 3) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     6.0 │       0.283185 │     NaN │        NaN │
│       2.0 │  6.2828 │    0.000389449 │ 6.22261 │   0.017149 │
│       3.0 │ 6.28319 │     4.24732e-9 │ 2.45542 │    71.9059 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘


In [ ]:
# | include: false 
# Yanting 
fn(x) = x^3 - x - 1
fn1(x) = 3x^2 - 1
fn2(x) = 6x

x0 = 1.0
ξ = 1.32471795724474602596090885447809734073440405690173336453401505030282785124

xo = osada(fn, fn1, fn2, 1, x0)
orderOfConvergence(xo, ξ, α = 2)

┌───────────┬─────────┬────────────────┬─────────┬────────────┐
│ iteration │    x[n] │ absolute error │   alpha │ mu (α = 2) │
├───────────┼─────────┼────────────────┼─────────┼────────────┤
│       1.0 │     1.0 │       0.324718 │     NaN │        NaN │
│       2.0 │     1.5 │       0.175282 │ 1.54815 │    1.66236 │
│       3.0 │ 1.34783 │      0.0231081 │ 2.16358 │   0.752125 │
│       4.0 │  1.3252 │    0.000482442 │ 2.02694 │   0.903473 │
│       5.0 │ 1.32472 │     2.16754e-7 │ 2.00932 │   0.931275 │
│       6.0 │ 1.32472 │    4.37428e-14 │ 2.00466 │   0.931046 │
│       7.0 │ 1.32472 │            0.0 │     Inf │        0.0 │
└───────────┴─────────┴────────────────┴─────────┴────────────┘
(fn(ξ), fn1(ξ), fn2(ξ)) = (2.220446049250313e-16, 4.264632998740079, 7.948307743468476)


(2.220446049250313e-16, 4.264632998740079, 7.948307743468476)

In [ ]:
# | include: false

#simon 

h = x -> x^3 - 8
h_prime = x -> 3x^2
w = Newton( h, h_prime, 100.)

orderOfConvergence( w, 2; α=2)

┌───────────┬─────────┬────────────────┬───────────┬────────────┐
│ iteration │    x[n] │ absolute error │     alpha │ mu (α = 2) │
├───────────┼─────────┼────────────────┼───────────┼────────────┤
│       1.0 │   100.0 │           98.0 │       NaN │        NaN │
│       2.0 │ 66.6669 │        64.6669 │   0.90933 │ 0.00673333 │
│       3.0 │ 44.4452 │        42.4452 │  0.899014 │    0.01015 │
│       4.0 │ 29.6315 │        27.6315 │  0.885477 │  0.0153372 │
│       5.0 │ 19.7574 │        17.7574 │  0.866779 │  0.0232579 │
│       6.0 │ 13.1784 │        11.1784 │  0.839121 │  0.0354505 │
│       7.0 │ 8.80096 │        6.80096 │  0.794149 │  0.0544265 │
│       8.0 │ 5.90174 │        3.90174 │   0.71016 │  0.0843562 │
│       9.0 │ 4.01105 │        2.01105 │  0.513183 │   0.132101 │
│      10.0 │ 2.83978 │       0.839784 │ -0.249923 │   0.207645 │
│      11.0 │ 2.22386 │       0.223862 │    8.5718 │   0.317428 │
│      12.0 │ 2.02178 │      0.0217786 │    2.5568 │    0.43458 │
│      13.

In [101]:
# | include: false 
# simon 

function Traub( h, h_prime, x1; N=100, tol=1e-10)
    x = [x1]
    for n in 2:N
        y = x[n-1] - h(x[n-1])/h_prime(x[n-1])
        push!( x, x[n-1] - (2h(x[n-1]))/(h_prime(x[n-1]) + h_prime(y)))
        r = abs(h(x[end]));
        if (r < tol)
            return x
        end
    end
    @warn "max interations |h| = $r";
    return x
end

h = x -> x^3 - 8
h_prime = x -> 3x^2
z = Traub(h, h_prime, 100.)

orderOfConvergence( z, 2; α=3)

┌───────────┬─────────┬────────────────┬───────────┬─────────────┐
│ iteration │    x[n] │ absolute error │     alpha │  mu (α = 3) │
├───────────┼─────────┼────────────────┼───────────┼─────────────┤
│       1.0 │   100.0 │           98.0 │       NaN │         NaN │
│       2.0 │ 53.8466 │        51.8466 │  0.861138 │  5.50861e-5 │
│       3.0 │  28.996 │         26.996 │  0.834713 │ 0.000193704 │
│       4.0 │  15.619 │         13.619 │  0.792388 │ 0.000692223 │
│       5.0 │    8.43 │           6.43 │  0.712617 │  0.00254553 │
│       6.0 │ 4.60695 │        2.60695 │  0.514881 │  0.00980617 │
│       7.0 │ 2.70353 │       0.703532 │ -0.366989 │   0.0397087 │
│       8.0 │  2.0505 │      0.0504967 │   8.49115 │    0.145015 │
│       9.0 │ 2.00004 │     3.55831e-5 │   3.43073 │    0.276347 │
│      10.0 │     2.0 │    1.33227e-14 │   3.11894 │    0.295707 │
└───────────┴─────────┴────────────────┴───────────┴─────────────┘
